# NB-01: マスターデータセット構築

**目的**: 2020〜2025年の全開催データから、すべての予測モデルの根幹となる学習母表を構築する。

## 参照仕様書
- `docs/html/modeling/horse_pre_race_dataset_spec.html` — 母表の定義 (BaselineV1)
- `docs/decisions/AREA-07-modeling.md` — モデリング管理正本
- `src/pipeline/models/layer_a_dataset.py` — 実装リファレンス

## 粒度・スパイン
```
1行 = 1レース × 1頭
主キー: race_id + horse_number → 後段で horse_id に正規化
```

## 結合ステップ
```
Step 0: Spine        ... race_shutuba_flat         (レース直前出走情報)
Step 1: Label        ... race_result_flat          (finish_position / time_sec)
Step 2: Speed Index  ... race_index_flat           (スピード指数・近3走展開)
Step 3: Past History ... race_shutuba_past_flat    (過去走サマリー・調教)
Step 4: JT Stats     ... jt_race_features          (騎手×調教師統計, 174 cols)
Step 5: Post-proc    ... 大衆指標削除・派生特徴量・型変換・分割
```

## 除外ポリシー
- **当日オッズ・人気は完全除外** (PUBLIC_INDICATOR_SET)
- **当日着順・タイムは Label のみ** (学習時のみ結合、推論時は使わない)
- **リーク防止**: JT統計は `as_of_race_id = 対象レース直前` で構築済み

## 0. セットアップ・定数定義

In [1]:
import sys, os
from pathlib import Path

# リポジトリルートを sys.path に追加
REPO_ROOT = Path("../../").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)
print("REPO_ROOT:", REPO_ROOT)
print("cwd:", Path.cwd())

REPO_ROOT: /home/jovyan/work/keiba-vpn
cwd: /home/jovyan/work/keiba-vpn


In [2]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import json
from typing import Optional

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 20)
pd.set_option("display.float_format", "{:.4f}".format)

# ─── パス定義 ───────────────────────────────────────────────────────────────
TABLES_DIR   = Path("data/page_reference/tables")
JT_STATS_DIR = Path("data/local/features/race_horse_tbl")
OUTPUT_DIR   = Path("data/local/modeling")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ─── 対象年 ─────────────────────────────────────────────────────────────────
# 2020–2025: Train=2020–2023 / Valid=2024 / Test=2025
TRAIN_YEARS = ["2020", "2021", "2022", "2023"]
VALID_YEARS = ["2024"]
TEST_YEARS  = ["2025"]
ALL_YEARS   = TRAIN_YEARS + VALID_YEARS + TEST_YEARS

print("ALL_YEARS:", ALL_YEARS)
print("OUTPUT_DIR:", OUTPUT_DIR)

ALL_YEARS: ['2020', '2021', '2022', '2023', '2024', '2025']
OUTPUT_DIR: data/local/modeling


In [3]:
# 大衆指標（当日オッズ・人気）除外リスト
# 参照: src/pipeline/features/feature_builder.py :: PUBLIC_INDICATOR_SET
from src.pipeline.features.feature_builder import PUBLIC_INDICATOR_SET, _drop_public_indicators
from src.pipeline.features.race_index_speed_recent import expand_speed_recent_in_dataframe

print(f"除外対象カラム数: {len(PUBLIC_INDICATOR_SET)}")
print("例:", sorted(list(PUBLIC_INDICATOR_SET))[:10])

除外対象カラム数: 41
例: ['fuku_odds', 'fukusho_odds', 'implied_place_prob', 'implied_win_prob', 'log_odds', 'market_odds', 'odds', 'odds_ratio', 'place_odds', 'place_odds_max']


## 1. データ読み込み関数

In [4]:
def load_flat(year: str, table: str, base_dir: Path = TABLES_DIR) -> Optional[pd.DataFrame]:
    """flat Parquet を読み込む。ファイルが存在しなければ None。"""
    p = base_dir / year / f"{table}_flat.parquet"
    if not p.exists():
        print(f"  [SKIP] {p} not found")
        return None
    df = pd.read_parquet(p)
    df["year"] = year
    return df


def load_jt_stats(year: str, base_dir: Path = JT_STATS_DIR) -> Optional[pd.DataFrame]:
    """騎手×調教師統計 (jt_race_features) を読み込む。"""
    p = base_dir / year / "jt_race_features.parquet"
    if not p.exists():
        print(f"  [SKIP] JT stats {p} not found")
        return None
    return pd.read_parquet(p)


# データ存在確認
for y in ALL_YEARS:
    shutuba = (TABLES_DIR / y / "race_shutuba_flat.parquet").exists()
    result  = (TABLES_DIR / y / "race_result_flat.parquet").exists()
    index   = (TABLES_DIR / y / "race_index_flat.parquet").exists()
    past    = (TABLES_DIR / y / "race_shutuba_past_flat.parquet").exists()
    jt      = (JT_STATS_DIR / y / "jt_race_features.parquet").exists()
    print(f"{y}: shutuba={shutuba}  result={result}  index={index}  past={past}  jt={jt}")

2020: shutuba=True  result=True  index=True  past=True  jt=True
2021: shutuba=True  result=True  index=True  past=True  jt=True
2022: shutuba=True  result=True  index=True  past=True  jt=True
2023: shutuba=True  result=True  index=True  past=True  jt=True
2024: shutuba=True  result=True  index=True  past=True  jt=True
2025: shutuba=True  result=True  index=True  past=True  jt=True


## 2. Step 0: Spine 構築 (race_shutuba_flat)

`race_shutuba_flat` を全年結合し、JRA 中央開催のみに絞る。  
当日の `odds` / `popularity` は後で削除する（この段階では保持して確認）。

In [5]:
print("=== Spine 読み込み (race_shutuba_flat) ===")
shutuba_parts = []
for y in ALL_YEARS:
    df = load_flat(y, "race_shutuba")
    if df is not None:
        shutuba_parts.append(df)
        print(f"  {y}: {len(df):,} 行")

spine = pd.concat(shutuba_parts, ignore_index=True)
print(f"\nSpine 結合後: {spine.shape}")

=== Spine 読み込み (race_shutuba_flat) ===
  2020: 48,282 行
  2021: 47,821 行
  2022: 47,220 行
  2023: 47,672 行
  2024: 47,181 行
  2025: 47,884 行

Spine 結合後: (286060, 34)


In [6]:
# date 列を datetime に変換
spine["race_date"] = pd.to_datetime(spine["date"], errors="coerce")

# JRA 中央開催のみ (venue_code 1-10)
def jra_mask(df: pd.DataFrame) -> pd.Series:
    if "venue_code" not in df.columns:
        return pd.Series(True, index=df.index)
    vc = pd.to_numeric(df["venue_code"], errors="coerce")
    return vc.between(1, 10, inclusive="both")

before = len(spine)
spine = spine[jra_mask(spine)].copy()
print(f"JRA フィルタ: {before:,} → {len(spine):,} 行")

# 重複除去 (race_id + horse_number)
spine = spine.drop_duplicates(subset=["race_id", "horse_number"])
print(f"重複除去後: {len(spine):,} 行")
print(f"レース数: {spine['race_id'].nunique():,}")
print(f"期間: {spine['race_date'].min().date()} 〜 {spine['race_date'].max().date()}")
spine.head(3)

JRA フィルタ: 286,060 → 286,060 行
重複除去後: 286,060 行
レース数: 20,733
期間: 2020-01-05 〜 2025-12-28


,race_id,date,venue,surface,distance,direction,grade,race_class,weather,track_condition,start_time,field_size,race_name,venue_code,round,weight_rule,course_type,horse_number,bracket_number,horse_name,horse_id,sex_age,jockey_weight,jockey_name,jockey_id,trainer_name,trainer_id,weight,weight_change,odds,popularity,sire,dam_sire,year,race_date
0,202001010101,2020-07-25,札幌,芝,1800,右,未勝利,サラ系２歳 未勝利,曇,良,09:55,6,2歳未勝利,01,1,馬齢,,1,1,ジュンブーケ,2018102410,牝2,52.0000,△亀田,01176,森,00427,442,0,0.0000,0,,,2020,2020-07-25
1,202001010101,2020-07-25,札幌,芝,1800,右,未勝利,サラ系２歳 未勝利,曇,良,09:55,6,2歳未勝利,01,1,馬齢,,2,2,アークライト,2018105193,牡2,54.0000,ルメール,05339,藤沢和,00386,510,0,0.0000,0,,,2020,2020-07-25
2,202001010101,2020-07-25,札幌,芝,1800,右,未勝利,サラ系２歳 未勝利,曇,良,09:55,6,2歳未勝利,01,1,馬齢,,3,3,ギャラントウォリア,2018104800,牡2,54.0000,池添,01032,平田,01082,482,-6,0.0000,0,,,2020,2020-07-25


## 3. Step 1: ラベル結合 (race_result_flat)

`finish_position` / `time_sec` / `last_3f` を Spine に結合する。  
**学習時のみ使用。推論パイプラインでは結合しない。**

In [7]:
print("=== ラベル読み込み (race_result_flat) ===")
result_parts = []
for y in ALL_YEARS:
    df = load_flat(y, "race_result")
    if df is not None:
        result_parts.append(df)
        print(f"  {y}: {len(df):,} 行")

result_all = pd.concat(result_parts, ignore_index=True)

# ラベルとして使う列のみ抽出
LABEL_COLS = ["race_id", "horse_number", "finish_position", "time_sec", "last_3f", "margin"]
result_label = result_all[[c for c in LABEL_COLS if c in result_all.columns]].copy()
result_label = result_label.drop_duplicates(subset=["race_id", "horse_number"])
print(f"\nラベル行数: {len(result_label):,}")

# finish_position を数値化
result_label["finish_position"] = pd.to_numeric(result_label["finish_position"], errors="coerce")
result_label["time_sec"] = pd.to_numeric(result_label["time_sec"], errors="coerce")

# 完走馬のみ (finish_position が 1-18 の整数値)
valid_finish = result_label["finish_position"].between(1, 18)
print(f"完走馬ラベル: {valid_finish.sum():,} 行 / 全 {len(result_label):,} 行")
result_label.head(3)

=== ラベル読み込み (race_result_flat) ===
  2020: 48,282 行
  2021: 47,821 行
  2022: 47,220 行
  2023: 47,672 行
  2024: 47,181 行
  2025: 47,884 行

ラベル行数: 286,060
完走馬ラベル: 283,714 行 / 全 286,060 行


,race_id,horse_number,finish_position,time_sec,last_3f,margin
0,202001010101,6,1,109.7000,35.6000,
1,202001010101,2,2,110.0000,35.8000,1.3/4
2,202001010101,3,3,110.1000,36.2000,1/2


In [8]:
# Spine にラベルを left join
spine = spine.merge(
    result_label,
    on=["race_id", "horse_number"],
    how="left",
    suffixes=("", "_result"),
)
print(f"ラベル結合後: {spine.shape}")
print(f"finish_position 取得率: {spine['finish_position'].notna().mean():.1%}")
print(f"time_sec 取得率: {spine['time_sec'].notna().mean():.1%}")

ラベル結合後: (286060, 39)
finish_position 取得率: 100.0%
time_sec 取得率: 100.0%


## 4. Step 2: スピード指数結合 (race_index_flat)

In [9]:
print("=== スピード指数読み込み (race_index_flat) ===")
INDEX_FEATURE_COLS = [
    "race_id", "horse_number",
    "speed_max", "speed_avg", "speed_distance", "speed_course", "speed_recent",
    "all_txt_c",
]

index_parts = []
for y in ALL_YEARS:
    df = load_flat(y, "race_index")
    if df is not None:
        # 存在する列だけ取る
        cols = [c for c in INDEX_FEATURE_COLS if c in df.columns]
        index_parts.append(df[cols])
        print(f"  {y}: {len(df):,} 行  cols={cols[2:]}")

index_all = pd.concat(index_parts, ignore_index=True)
index_all = index_all.drop_duplicates(subset=["race_id", "horse_number"])
print(f"\n指数行数: {len(index_all):,}")

=== スピード指数読み込み (race_index_flat) ===
  2020: 22,918 行  cols=['speed_max', 'speed_avg', 'speed_distance', 'speed_course', 'speed_recent', 'all_txt_c']
  2021: 21,811 行  cols=['speed_max', 'speed_avg', 'speed_distance', 'speed_course', 'speed_recent', 'all_txt_c']
  2022: 22,484 行  cols=['speed_max', 'speed_avg', 'speed_distance', 'speed_course', 'speed_recent', 'all_txt_c']
  2023: 22,805 行  cols=['speed_max', 'speed_avg', 'speed_distance', 'speed_course', 'speed_recent', 'all_txt_c']
  2024: 23,980 行  cols=['speed_max', 'speed_avg', 'speed_distance', 'speed_course', 'speed_recent', 'all_txt_c']
  2025: 23,522 行  cols=['speed_max', 'speed_avg', 'speed_distance', 'speed_course', 'speed_recent', 'all_txt_c']

指数行数: 137,520


In [10]:
# speed_recent リストを 3列に展開
if "speed_recent" in index_all.columns:
    index_all = expand_speed_recent_in_dataframe(index_all, col="speed_recent", dest_prefix="speed_recent")
    print("speed_recent → speed_recent_1/2/3 に展開完了")
    print(index_all[["race_id","horse_number","speed_recent_1","speed_recent_2","speed_recent_3"]].head(3))
else:
    print("speed_recent 列なし")

# Spine に結合
# race_shutuba_flat にも time_index_m 等がある場合は suffix で区別
spine = spine.merge(
    index_all,
    on=["race_id", "horse_number"],
    how="left",
    suffixes=("", "_idx"),
)
print(f"指数結合後: {spine.shape}")
print(f"speed_max 取得率: {spine['speed_max'].notna().mean():.1%}" if 'speed_max' in spine else "")

speed_recent → speed_recent_1/2/3 に展開完了
        race_id  horse_number  speed_recent_1  speed_recent_2  speed_recent_3
0  202001010101        1.0000         64.0000            <NA>            <NA>
1  202001010101        2.0000         63.0000            <NA>            <NA>
2  202001010101        3.0000         76.0000            <NA>            <NA>
指数結合後: (286060, 47)
speed_max 取得率: 47.4%


## 5. Step 3: 過去走サマリー結合 (race_shutuba_past_flat)

In [11]:
print("=== 過去走サマリー読み込み (race_shutuba_past_flat) ===")
PAST_FEATURE_COLS = ["race_id", "horse_number", "past_races", "training"]

past_parts = []
for y in ALL_YEARS:
    df = load_flat(y, "race_shutuba_past")
    if df is not None:
        cols = [c for c in PAST_FEATURE_COLS if c in df.columns]
        past_parts.append(df[cols])
        print(f"  {y}: {len(df):,} 行")

past_all = pd.concat(past_parts, ignore_index=True)
past_all = past_all.drop_duplicates(subset=["race_id", "horse_number"])
print(f"\n過去走行数: {len(past_all):,}")

spine = spine.merge(
    past_all,
    on=["race_id", "horse_number"],
    how="left",
    suffixes=("", "_past"),
)
print(f"過去走結合後: {spine.shape}")

=== 過去走サマリー読み込み (race_shutuba_past_flat) ===
  2020: 48,282 行
  2021: 47,821 行
  2022: 47,220 行
  2023: 47,672 行
  2024: 47,181 行
  2025: 47,884 行

過去走行数: 285,890
過去走結合後: (286060, 49)


## 6. Step 4: 騎手×調教師統計結合 (jt_race_features)

`jt_race_features.parquet` は `race_id + horse_id` キー。  
Spine には `horse_id` があるため、`race_id + horse_id` で結合。

In [12]:
print("=== 騎手×調教師統計読み込み (jt_race_features) ===")
jt_parts = []
for y in ALL_YEARS:
    df = load_jt_stats(y)
    if df is not None:
        jt_parts.append(df)
        print(f"  {y}: {len(df):,} 行  cols={df.shape[1]}")

if jt_parts:
    jt_all = pd.concat(jt_parts, ignore_index=True)
    jt_all = jt_all.drop_duplicates(subset=["race_id", "horse_id"])
    print(f"\nJT行数: {len(jt_all):,}  cols={jt_all.shape[1]}")
    
    # メタ列を除外（結合キー以外の文字列・日付は除く）
    JT_DROP = ["jt_result_date", "jt_race_datetime", "jt_row_jockey_id", "jt_row_trainer_id"]
    jt_all = jt_all.drop(columns=[c for c in JT_DROP if c in jt_all.columns])
    
    spine = spine.merge(
        jt_all,
        on=["race_id", "horse_id"],
        how="left",
        suffixes=("", "_jt"),
    )
    print(f"JT結合後: {spine.shape}")
    # 取得率
    jt_numeric_cols = [c for c in jt_all.columns if c not in ("race_id","horse_id") and jt_all[c].dtype in ("float64","int64")]
    if jt_numeric_cols:
        rate = spine[jt_numeric_cols[0]].notna().mean()
        print(f"JT取得率 ({jt_numeric_cols[0]}): {rate:.1%}")
else:
    print("JT統計なし（jt_race_features が未生成の可能性）")

=== 騎手×調教師統計読み込み (jt_race_features) ===
  2020: 47,876 行  cols=174
  2021: 47,476 行  cols=174
  2022: 46,840 行  cols=174
  2023: 47,273 行  cols=174
  2024: 46,752 行  cols=174
  2025: 47,497 行  cols=174

JT行数: 283,714  cols=174
JT結合後: (286060, 217)
JT取得率 (jk_at_dist_avg_finish): 98.5%


## 7. Step 5: 後処理
### 7.1 大衆指標削除

In [13]:
before_cols = spine.shape[1]
spine = _drop_public_indicators(spine)
print(f"大衆指標削除: {before_cols} → {spine.shape[1]} 列")

大衆指標削除: 217 → 215 列


### 7.2 数値型変換・基本的な特徴量エンジニアリング

In [14]:
# ── 性齢の分割 ──────────────────────────────────────────────────────────────
if "sex_age" in spine.columns:
    spine["sex"]  = spine["sex_age"].str.extract(r"([牡牝セ騸])", expand=False)
    spine["age"]  = pd.to_numeric(spine["sex_age"].str.extract(r"(\d+)", expand=False), errors="coerce")
    print("sex_age 分割: sex, age")

# ── 数値変換 ────────────────────────────────────────────────────────────────
for col in ["jockey_weight", "weight", "weight_change", "distance", "field_size",
            "bracket_number", "horse_number"]:
    if col in spine.columns:
        spine[col] = pd.to_numeric(spine[col], errors="coerce")

# ── カテゴリエンコード（ラベル → 整数コード）──────────────────────────────
CAT_COLS = ["venue", "surface", "direction", "weather", "track_condition",
            "grade", "race_class", "weight_rule", "course_type", "sex"]
for col in CAT_COLS:
    if col in spine.columns:
        spine[col] = spine[col].astype("category")

# ── 欠損率サマリー ───────────────────────────────────────────────────────────
missing = spine.isnull().mean().sort_values(ascending=False)
high_missing = missing[missing > 0.5]
print(f"\n欠損率 > 50% の列: {len(high_missing)}")
if len(high_missing) > 0:
    print(high_missing.head(10))

print(f"\nDataFrame 概観:")
print(spine.describe(include="all").T[["count","mean","std","min","max"]].head(20))

sex_age 分割: sex, age

欠損率 > 50% の列: 8
speed_recent_3   0.6477
speed_recent_2   0.6031
speed_recent_1   0.5520
all_txt_c        0.5369
speed_max        0.5256
speed_distance   0.5256
speed_course     0.5256
speed_avg        0.5256
dtype: float64

DataFrame 概観:
                      count      mean      std       min       max
race_id              286060       NaN      NaN       NaN       NaN
date                 286060       NaN      NaN       NaN       NaN
venue                286060       NaN      NaN       NaN       NaN
surface              286060       NaN      NaN       NaN       NaN
distance        286060.0000 1657.2287 420.0501 1000.0000 4260.0000
direction            286060       NaN      NaN       NaN       NaN
grade                286060       NaN      NaN       NaN       NaN
race_class           286060       NaN      NaN       NaN       NaN
weather              286060       NaN      NaN       NaN       NaN
track_condition      286060       NaN      NaN       NaN       NaN
sta

### 7.3 ターゲット変数の確認

In [15]:
import matplotlib
matplotlib.use("Agg")  # headless 環境対応
import matplotlib.pyplot as plt

# 完走フラグ
spine["is_finisher"] = spine["finish_position"].between(1, 18)

# 1着フラグ（T-1: 勝率モデルのターゲット）
spine["is_winner"] = (spine["finish_position"] == 1).astype(int)

# 3着内フラグ（複勝ターゲット）
spine["is_top3"] = spine["finish_position"].between(1, 3).astype(int)

# 年別・ターゲット分布
print("=== ターゲット分布 ===")
print(spine.groupby("year")[["is_finisher","is_winner","is_top3"]].mean().round(3))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# finish_position 分布
fp = spine["finish_position"].dropna().astype(int)
fp[fp <= 18].value_counts().sort_index().plot(kind="bar", ax=axes[0], color="steelblue")
axes[0].set_title("finish_position 分布")
axes[0].set_xlabel("着順")

# 年別レース数
spine.groupby("year")["race_id"].nunique().plot(kind="bar", ax=axes[1], color="tomato")
axes[1].set_title("年別 レース数")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "target_distribution.png", dpi=120)
plt.show()
print(f"図を保存: {OUTPUT_DIR / 'target_distribution.png'}")

=== ターゲット分布 ===
      is_finisher  is_winner  is_top3
year                                 
2020       0.9920     0.0720   0.2150
2021       0.9930     0.0720   0.2170
2022       0.9920     0.0730   0.2200
2023       0.9920     0.0730   0.2180
2024       0.9910     0.0730   0.2200
2025       0.9920     0.0720   0.2170
図を保存: data/local/modeling/target_distribution.png


### 7.4 Train / Valid / Test 分割フラグ付与

仕様: `docs/html/modeling/train_valid_test_split_strategy.html`  
- Train: 2020–2023  
- Valid: 2024  
- Test: 2025  
- **ランダム split 禁止** (時系列 Purged GroupKFold)

In [16]:
def assign_split(year_str: str) -> str:
    if year_str in TRAIN_YEARS:
        return "train"
    elif year_str in VALID_YEARS:
        return "valid"
    elif year_str in TEST_YEARS:
        return "test"
    return "other"

spine["split"] = spine["year"].map(assign_split)

split_counts = spine.groupby(["split","year"])["race_id"].agg(["count","nunique"])
split_counts.columns = ["行数","レース数"]
print(split_counts)
print("\n分割サマリー:")
print(spine["split"].value_counts())

               行数  レース数
split year             
test  2025  47884  3455
train 2020  48282  3456
      2021  47821  3456
      2022  47220  3456
      2023  47672  3456
valid 2024  47181  3454

分割サマリー:
split
train    190995
test      47884
valid     47181
Name: count, dtype: int64


## 8. データセット保存

In [17]:
# ── 列の整理 ────────────────────────────────────────────────────────────────
# 重複列の処理 (suffix付き列は削除)
dup_suffix_cols = [c for c in spine.columns if c.endswith("_result") or c.endswith("_idx") or c.endswith("_past")]
# ただし元列が存在しない suffix 列は残す
drop_dup = [c for c in dup_suffix_cols if c.replace("_result","").replace("_idx","").replace("_past","") in spine.columns]
if drop_dup:
    print(f"重複列を削除: {drop_dup}")
    spine = spine.drop(columns=drop_dup, errors="ignore")

print(f"\n最終 shape: {spine.shape}")
print(f"列数: {spine.shape[1]}")

# ── 分割別に保存 ─────────────────────────────────────────────────────────────
for split_name in ["train", "valid", "test"]:
    df_split = spine[spine["split"] == split_name].copy()
    out_path = OUTPUT_DIR / f"master_dataset_{split_name}.parquet"
    df_split.to_parquet(out_path, index=False)
    print(f"保存: {out_path}  shape={df_split.shape}")

# ── 全データも保存 ───────────────────────────────────────────────────────────
full_path = OUTPUT_DIR / "master_dataset_full.parquet"
spine.to_parquet(full_path, index=False)
print(f"\n全データ保存: {full_path}  shape={spine.shape}")


最終 shape: (286060, 221)
列数: 221
保存: data/local/modeling/master_dataset_train.parquet  shape=(190995, 221)
保存: data/local/modeling/master_dataset_valid.parquet  shape=(47181, 221)
保存: data/local/modeling/master_dataset_test.parquet  shape=(47884, 221)

全データ保存: data/local/modeling/master_dataset_full.parquet  shape=(286060, 221)


In [18]:
# ── メタ情報を JSON で保存 ──────────────────────────────────────────────────
meta = {
    "created_at": pd.Timestamp.now().isoformat(),
    "years": ALL_YEARS,
    "train_years": TRAIN_YEARS,
    "valid_years": VALID_YEARS,
    "test_years": TEST_YEARS,
    "total_rows": int(len(spine)),
    "total_cols": int(spine.shape[1]),
    "total_races": int(spine["race_id"].nunique()),
    "split_counts": spine["split"].value_counts().to_dict(),
    "col_list": list(spine.columns),
    "label_coverage": {
        "finish_position": float(spine["finish_position"].notna().mean()),
        "time_sec": float(spine["time_sec"].notna().mean()),
    },
}

meta_path = OUTPUT_DIR / "master_dataset.meta.json"
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)
print(f"メタ保存: {meta_path}")
print(json.dumps({k: v for k, v in meta.items() if k != "col_list"}, ensure_ascii=False, indent=2))

メタ保存: data/local/modeling/master_dataset.meta.json
{
  "created_at": "2026-07-10T12:20:30.829134",
  "years": [
    "2020",
    "2021",
    "2022",
    "2023",
    "2024",
    "2025"
  ],
  "train_years": [
    "2020",
    "2021",
    "2022",
    "2023"
  ],
  "valid_years": [
    "2024"
  ],
  "test_years": [
    "2025"
  ],
  "total_rows": 286060,
  "total_cols": 221,
  "total_races": 20733,
  "split_counts": {
    "train": 190995,
    "test": 47884,
    "valid": 47181
  },
  "label_coverage": {
    "finish_position": 1.0,
    "time_sec": 1.0
  }
}


## 9. データセット品質チェック

In [19]:
print("=== 品質チェック ===")

# 1. リーク確認: train の race_id が valid/test に存在しないこと
train_races = set(spine[spine["split"]=="train"]["race_id"])
valid_races  = set(spine[spine["split"]=="valid"]["race_id"])
test_races   = set(spine[spine["split"]=="test"]["race_id"])
leak_tv = train_races & valid_races
leak_tt = train_races & test_races
print(f"[{'OK' if not leak_tv else 'NG'}] train ∩ valid = {len(leak_tv)}")
print(f"[{'OK' if not leak_tt else 'NG'}] train ∩ test  = {len(leak_tt)}")

# 2. 主キー重複確認
dup_keys = spine.duplicated(subset=["race_id","horse_number"]).sum()
print(f"[{'OK' if dup_keys==0 else 'NG'}] 主キー重複: {dup_keys}")

# 3. 完走ラベル取得率
label_rate = spine[spine["is_finisher"]]["finish_position"].notna().mean()
print(f"[{'OK' if label_rate > 0.9 else 'WARN'}] ラベル取得率 (完走馬): {label_rate:.1%}")

# 4. 公開指標の混入確認
leak_cols = [c for c in spine.columns if c in PUBLIC_INDICATOR_SET]
print(f"[{'OK' if not leak_cols else 'NG'}] 大衆指標混入: {leak_cols}")

# 5. 各年のレース数が妥当か
race_per_year = spine.groupby("year")["race_id"].nunique()
print(f"\n年別レース数 (JRA 中央: 年1700〜1900 が目安)")
print(race_per_year)

=== 品質チェック ===
[OK] train ∩ valid = 0
[OK] train ∩ test  = 0
[OK] 主キー重複: 0
[OK] ラベル取得率 (完走馬): 100.0%
[OK] 大衆指標混入: []

年別レース数 (JRA 中央: 年1700〜1900 が目安)
year
2020    3456
2021    3456
2022    3456
2023    3456
2024    3454
2025    3455
Name: race_id, dtype: int64


## 10. 特徴量カタログ

In [20]:
# カラムをソースブロック別に分類して表示
META_COLS   = ["race_id", "horse_id", "horse_number", "horse_name", "jockey_id", "trainer_id",
               "date", "race_date", "year", "split", "round", "venue_code"]
LABEL_COLS2 = ["finish_position", "time_sec", "last_3f", "margin",
               "is_finisher", "is_winner", "is_top3"]
RACE_COLS   = ["venue", "surface", "distance", "direction", "weather", "track_condition",
               "grade", "race_class", "weight_rule", "course_type", "field_size",
               "race_name", "start_time"]
HORSE_COLS  = ["sex_age", "sex", "age", "weight", "weight_change",
               "bracket_number", "horse_number", "jockey_name", "trainer_name"]
INDEX_COLS  = [c for c in spine.columns if c.startswith("speed") or c in ("all_txt_c",)]
JT_COLS     = [c for c in spine.columns if c.startswith("jk_") or c.startswith("tr_") or c.startswith("jt_")]
OTHER_COLS  = [c for c in spine.columns
               if c not in set(META_COLS+LABEL_COLS2+RACE_COLS+HORSE_COLS+INDEX_COLS+JT_COLS)]

catalog = {
    "META":   META_COLS,
    "LABEL":  LABEL_COLS2,
    "RACE":   RACE_COLS,
    "HORSE":  HORSE_COLS,
    "INDEX":  INDEX_COLS,
    "JT":     JT_COLS[:20],  # 174列あるので先頭20だけ表示
    "OTHER":  OTHER_COLS[:20],
}

for block, cols in catalog.items():
    avail = [c for c in cols if c in spine.columns]
    print(f"[{block:6}] {len(avail):3d} 列: {avail[:8]}")

[META  ]  12 列: ['race_id', 'horse_id', 'horse_number', 'horse_name', 'jockey_id', 'trainer_id', 'date', 'race_date']
[LABEL ]   7 列: ['finish_position', 'time_sec', 'last_3f', 'margin', 'is_finisher', 'is_winner', 'is_top3']
[RACE  ]  13 列: ['venue', 'surface', 'distance', 'direction', 'weather', 'track_condition', 'grade', 'race_class']
[HORSE ]   9 列: ['sex_age', 'sex', 'age', 'weight', 'weight_change', 'bracket_number', 'horse_number', 'jockey_name']
[INDEX ]   8 列: ['speed_max', 'speed_avg', 'speed_distance', 'speed_course', 'all_txt_c', 'speed_recent_1', 'speed_recent_2', 'speed_recent_3']
[JT    ]  20 列: ['jk_at_dist_avg_finish', 'jk_at_dist_starts', 'jk_at_dist_top3', 'jk_at_dist_top3_rate', 'jk_at_dist_win_rate', 'jk_at_dist_wins', 'jk_at_grade_avg_finish', 'jk_at_grade_starts']
[OTHER ]   5 列: ['jockey_weight', 'sire', 'dam_sire', 'past_races', 'training']


## 11. 次ステップ

| ノートブック | 内容 |
|---|---|
| `nb-02-feature-engineering.ipynb` | 過去走展開・派生特徴量・OOF ターゲットエンコーディング |
| `nb-03-baseline-train.ipynb` | LightGBM 二値分類 (T-1) + Purged GroupKFold |
| `nb-04-multihead-train.ipynb` | 多段階パイプライン (T-4/T-6/T-8) |
| `nb-05-ensemble.ipynb` | LGBM + XGB + CatBoost スタッキング |
| `nb-06-evaluation.ipynb` | AUC/NDCG/ROI バックテスト |

### 現時点の成果物
```
data/local/modeling/
├── master_dataset_full.parquet   # 全期間 (2020–2025)
├── master_dataset_train.parquet  # 2020–2023
├── master_dataset_valid.parquet  # 2024
├── master_dataset_test.parquet   # 2025
├── master_dataset.meta.json      # メタ情報
└── target_distribution.png       # ターゲット分布図
```